# UTS Praktikum Kecerdasan Buatan — Template Notebook

Notebook ini mengikuti struktur soal pada PDF UTS: EDA, Linear Regression, serta K-NN (NIM ganjil) atau K-Means (NIM genap).

**Sebelum menjalankan:**
- Simpan dataset terkait di folder yang sama dengan notebook.
- Untuk Soal 2, periksa nama kolom pada `Sales_Data.csv` lalu sesuaikan variabel `PROMO_COL` dan `SALES_COL`.
- Pilih cabang Soal 3 sesuai NIM.
- Dataset `users_data_cleaned.pickle` dan `konsumen.csv` belum tersedia bersama soal; notebook ini tidak membuat-buat data tersebut.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## Soal 1 — Exploratory Data Analysis (EDA)

Dataset: `users_data_cleaned.pickle`

In [2]:
# 1. Load and preview the data
df = pd.read_pickle("users_data_cleaned.pickle")
display(df.head())


FileNotFoundError: [Errno 2] No such file or directory: 'users_data_cleaned.pickle'

In [3]:
# 2. Check the shape of the data
print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


NameError: name 'df' is not defined

In [ ]:
# 3. Check the data types
print(df.dtypes)


In [ ]:
# 4. Check for correlations
numeric_df = df.select_dtypes(include="number")
display(numeric_df.corr())


In [ ]:
# 5. Summary statistics for numerical data only
display(df.describe(include="number"))


In [ ]:
# 6. Histograms for all numerical columns
numeric_df.hist(figsize=(15, 10), bins=20)
plt.suptitle("Histograms of Numerical Columns")
plt.tight_layout()
plt.show()


In [ ]:
# 7. Box plots for age and number_transactions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, y="age", ax=axes[0])
axes[0].set_title("Box Plot: age")
sns.boxplot(data=df, y="number_transactions", ax=axes[1])
axes[1].set_title("Box Plot: number_transactions")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Line plot: date_joined to total_amount_usd
df["date_joined"] = pd.to_datetime(df["date_joined"])
line_data = (
    df.groupby("date_joined", as_index=False)["total_amount_usd"]
      .sum()
      .sort_values("date_joined")
)
sns.lineplot(data=line_data, x="date_joined", y="total_amount_usd", marker="o")
plt.title("Date Joined vs Total Amount USD")
plt.xlabel("Date Joined")
plt.ylabel("Total Amount USD")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 9. Scatter plot: total_amount_usd vs number_transactions
sns.scatterplot(data=df, x="number_transactions", y="total_amount_usd")
plt.title("Total Amount USD vs Number Transactions")
plt.tight_layout()
plt.show()


In [ ]:
# 10. Horizontal bar chart for job
df["job"].value_counts().sort_values().plot(kind="barh")
plt.title("Customers by Job")
plt.xlabel("Count")
plt.ylabel("Job")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Horizontal bar chart for marital
df["marital"].value_counts().sort_values().plot(kind="barh")
plt.title("Customers by Marital Status")
plt.xlabel("Count")
plt.ylabel("Marital")
plt.tight_layout()
plt.show()


In [ ]:
# 12. Heatmap for correlations
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# 13. Identify missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())


## Soal 2 — Linear Regression

Dataset: `Sales_Data.csv`

Periksa output kolom terlebih dahulu. Sesuaikan `PROMO_COL` dan `SALES_COL` dengan nama kolom yang benar-benar ada di CSV.

In [ ]:
# 1a. Read Sales_Data.csv and inspect variables
df_sales = pd.read_csv("Sales_Data.csv")
display(df_sales.head())
print("Columns:", df_sales.columns.tolist())
df_sales.info()


In [ ]:
# 1b. Define features and target (EDIT THESE COLUMN NAMES IF NEEDED)
PROMO_COL = "Biaya Promo"
SALES_COL = "Nilai Penjualan"

X = df_sales[[PROMO_COL]]
y = df_sales[SALES_COL]

display(X.head())
display(y.head())


In [ ]:
# 1c. Split data: train 80%, test 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
display(X_train.head())
display(X_test.head())


In [ ]:
# 1d. Call and train LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
print("Linear Regression model trained.")


In [ ]:
# 1e. Predict test data
y_pred = model.predict(X_test)
results = pd.DataFrame({
    "Biaya Promo": X_test[PROMO_COL].values,
    "Penjualan Aktual": y_test.values,
    "Penjualan Prediksi": y_pred
})
display(results)


In [ ]:
# 2. Intercept, slope, and R-Square
print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])
print("R-Square (test):", model.score(X_test, y_test))


In [ ]:
# 1f-1g. Plot actual train/test data and fitted regression line
plt.figure(figsize=(10, 6))
plt.scatter(X_train[PROMO_COL], y_train, label="Train actual")
plt.scatter(X_test[PROMO_COL], y_test, label="Test actual")

x_min = df_sales[PROMO_COL].min()
x_max = df_sales[PROMO_COL].max()
x_line = np.linspace(x_min, x_max, 200).reshape(-1, 1)
x_line_df = pd.DataFrame(x_line, columns=[PROMO_COL])
plt.plot(x_line, model.predict(x_line_df), label="Regression prediction")

plt.title("Promotion Cost vs Sales")
plt.xlabel(PROMO_COL)
plt.ylabel(SALES_COL)
plt.legend()
plt.tight_layout()
plt.show()


## Soal 3A — K-NN (khusus NIM ganjil)

Dataset: `pima-indians-diabetes.data.csv`

Jalankan bagian ini jika NIM ganjil.

In [ ]:
# 1. Read dataset; source data has no header
diabetes_columns = [
    "preg", "glukosa", "pres", "skin", "insulin", "BMI", "pedi", "age", "class"
]
df_diabetes = pd.read_csv(
    "pima-indians-diabetes.data.csv",
    header=None,
    names=diabetes_columns
)

# 2. Display first 8 rows
display(df_diabetes.head(8))


In [ ]:
# 3. Features and label; display first 9 rows
X_diabetes = df_diabetes.drop("class", axis=1)
y_diabetes = df_diabetes["class"]
display(X_diabetes.head(9))
display(y_diabetes.head(9))


In [ ]:
# 4. Split data: train 85%, test 15%
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes, y_diabetes, test_size=0.15, random_state=42
)
print("Training rows:", len(X_train_d))
print("Testing rows:", len(X_test_d))


In [ ]:
# 5-6. K-NN, k=5, Euclidean distance, and accuracy
knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
knn.fit(X_train_d, y_train_d)
y_pred_d = knn.predict(X_test_d)
print("Accuracy:", accuracy_score(y_test_d, y_pred_d))


In [ ]:
# 7. Accuracy for k=1..10
k_values = range(1, 11)
accuracies = []
for k in k_values:
    model_k = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    model_k.fit(X_train_d, y_train_d)
    pred_k = model_k.predict(X_test_d)
    accuracies.append(accuracy_score(y_test_d, pred_k))
    print(f"k={k}: accuracy={accuracies[-1]:.4f}")

plt.plot(list(k_values), accuracies, marker="o")
plt.title("K vs Accuracy (K-NN)")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.xticks(list(k_values))
plt.grid(True)
plt.tight_layout()
plt.show()


## Soal 3B — K-Means (khusus NIM genap)

Dataset: `konsumen.csv`

Jalankan bagian ini jika NIM genap. Jangan jalankan cabang yang tidak sesuai dengan NIM.

In [ ]:
# 1. Read dataset and display first 7 rows
df_konsumen = pd.read_csv("konsumen.csv")
display(df_konsumen.head(7))
print("Columns:", df_konsumen.columns.tolist())


In [ ]:
# 2-3. Drop No column and display first 9 rows
df_konsumen = df_konsumen.drop(columns=["No"])
display(df_konsumen.head(9))


In [ ]:
# 4. Plot salary vs spending
plt.scatter(df_konsumen["Gaji"], df_konsumen["Pengeluaran"])
plt.title("Gaji vs Pengeluaran")
plt.xlabel("Gaji")
plt.ylabel("Pengeluaran")
plt.tight_layout()
plt.show()


In [ ]:
# 5-6. Fit K-Means with 2 clusters and display cluster centers
X_konsumen = df_konsumen[["Gaji", "Pengeluaran"]]
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_konsumen["Cluster"] = kmeans.fit_predict(X_konsumen)

print("Pusat cluster:")
print(kmeans.cluster_centers_)
display(df_konsumen.head(9))


In [ ]:
# 7-8. Plot consumers in two different colors
sns.scatterplot(
    data=df_konsumen, x="Gaji", y="Pengeluaran",
    hue="Cluster", palette="Set1", s=100
)
plt.title("Hasil Clustering Konsumen")
plt.tight_layout()
plt.show()


In [ ]:
# 9. Plot clusters with cluster centers
centers = kmeans.cluster_centers_
sns.scatterplot(
    data=df_konsumen, x="Gaji", y="Pengeluaran",
    hue="Cluster", palette="Set1", s=100
)
plt.scatter(
    centers[:, 0], centers[:, 1],
    marker="X", s=250, color="black", label="Pusat Cluster"
)
plt.title("Clustering Konsumen dengan Pusat Cluster")
plt.legend()
plt.tight_layout()
plt.show()
